# Ngày 16 · Bagging, Random Forest và lỗi out-of-bag

- Lấy mẫu bootstrap và xác định các mẫu out-of-bag (OOB)
- Ghép nhiều cây trên mẫu bootstrap, mỗi lần tách chỉ xét √d cột
- Ước lượng lỗi bằng OOB mà không cần tập validation riêng

**Đọc:**
- ISLR (bản 2), §8.2.1 Bagging (đoạn Out-of-Bag Error Estimation) và §8.2.2 Random Forests
- Breiman (2001), Random Forests, Machine Learning 45(1), 5–32
- Hastie, Tibshirani & Friedman, ESL, §8.7 Bagging và Chương 15 Random Forests

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# Breast Cancer Wisconsin (Diagnostic), UCI, có sẵn trong scikit-learn. Cùng phép chia với tuần 2.
_bc = load_breast_cancer()
X_bc, y_bc = _bc.data, 1 - _bc.target   # 1 = ác tính
idx_tr, idx_va = train_test_split(np.arange(len(y_bc)), test_size=0.3, random_state=0, stratify=y_bc)
_scaler = StandardScaler().fit(X_bc[idx_tr])
X_tr, X_va = _scaler.transform(X_bc[idx_tr]), _scaler.transform(X_bc[idx_va])
y_tr, y_va = y_bc[idx_tr], y_bc[idx_va]
from sklearn.tree import DecisionTreeClassifier
def _bootstrap_ref(n, rng):
    idx = rng.integers(0, n, n)
    return idx, np.setdiff1d(np.arange(n), idx)
def _ref_oob(forest, X, y):
    votes, count = np.zeros(len(y)), np.zeros(len(y))
    for tree, oob in forest:
        votes[oob] += tree.predict(X[oob]); count[oob] += 1
    m = count > 0
    return float(np.mean((votes[m] / count[m] >= .5).astype(int) == y[m]))
def _forest_shape_ok(forest, n, n_trees):
    sizes = [len(o) for _, o in forest]
    return (len(forest) == n_trees and all(.25 * n < s < .5 * n for s in sizes)
            and len({tuple(o) for _, o in forest}) > 1)


## Mẫu bootstrap và chỉ số out-of-bag

Gợi ý: rng.integers(0, n, n) lấy n chỉ số có hoàn lại. OOB là các chỉ số trong 0..n−1 không xuất hiện: np.setdiff1d.

In [ ]:
def bootstrap_indices(n, rng):
    # rng là np.random.Generator. Trả (idx, oob): idx dài n (có hoàn lại), oob là các chỉ số không được chọn, đã sắp xếp.
    raise NotImplementedError


In [ ]:
_report('d16-bootstrap',
    _check('Có hoàn lại: khoảng 63,2% chỉ số khác nhau', lambda: ((lambda i: len(i) == 10000 and i.min() >= 0 and i.max() < 10000 and .62 < len(np.unique(i)) / 10000 < .65)(np.asarray(bootstrap_indices(10000, np.random.default_rng(1))[0])), 'n = 10000, kỳ vọng 1 − e⁻¹ ≈ 0,632.')),
    _check('OOB đúng bằng phần bù của idx', lambda: ((lambda r: np.array_equal(np.asarray(r[1]), np.setdiff1d(np.arange(50), r[0])))(bootstrap_indices(50, np.random.default_rng(2))), 'n = 50.')),
    _check('Dùng đúng rng được truyền vào', lambda: ((lambda a, b: np.array_equal(a[0], b[0]))(bootstrap_indices(30, np.random.default_rng(7)), bootstrap_indices(30, np.random.default_rng(7))) and not np.array_equal(*(lambda r: (bootstrap_indices(30, r)[0], bootstrap_indices(30, r)[0]))(np.random.default_rng(7))), 'Cùng seed thì giống nhau; hai lần gọi trên cùng rng thì khác nhau.')),
)


## Random Forest tự ghép và accuracy OOB

Gợi ý: Mỗi cây i: lấy bootstrap bằng một rng = np.random.default_rng(seed) tạo một lần ở đầu, rồi DecisionTreeClassifier(max_features=max_features, random_state=seed + i). Lưu cặp (cây, oob). OOB: mỗi mẫu chỉ nhận phiếu từ cây mà mẫu đó nằm trong oob.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

def fit_forest(X, y, n_trees, max_features, seed):
    # Trả list n_trees cặp (cây đã fit, mảng chỉ số oob). Dùng bootstrap_indices của bài trên.
    raise NotImplementedError

def predict_forest(forest, X):
    # Bỏ phiếu đa số; tỉ lệ phiếu lớp 1 >= 0,5 thì dự đoán 1.
    raise NotImplementedError

def oob_accuracy(forest, X, y):
    # Accuracy trên các mẫu có ít nhất một cây không thấy chúng.
    raise NotImplementedError


In [ ]:
bootstrap_indices = globals().get('bootstrap_indices') or _bootstrap_ref
_c16 = {}
def _f16():
    if 'f' not in _c16:
        _c16['f'] = fit_forest(X_tr, y_tr, 50, 'sqrt', 0)
    return _c16['f']
_report('d16-forest',
    _check('50 cây, mỗi cây có tập OOB riêng cỡ khoảng 37%', lambda: (_forest_shape_ok(_f16(), len(y_tr), 50), 'Kiểm tra cấu trúc (cây, oob).')),
    _check('Bỏ phiếu đa số của mọi cây (tỉ lệ ≥ 0,5 → 1)', lambda: (np.array_equal(predict_forest(_f16(), X_va), (np.mean([t.predict(X_va) for t, _ in _f16()], axis=0) >= .5).astype(int)), 'So với phiếu của 50 cây trên validation.')),
    _check('Cả rừng tốt hơn trung bình từng cây ít nhất 1 điểm %', lambda: ((lambda f, ind: f >= ind + .01 and f >= .93)(np.mean(predict_forest(_f16(), X_va) == y_va), np.mean([np.mean(t.predict(X_va) == y_va) for t, _ in _f16()])), 'Trên validation, max_features = "sqrt".')),
    _check('Accuracy OOB khớp cách tính tham chiếu', lambda: (np.isclose(oob_accuracy(_f16(), X_tr, y_tr), _ref_oob(_f16(), X_tr, y_tr)), 'Chỉ cây không thấy mẫu mới được bỏ phiếu cho mẫu đó.')),
)


## Dự án

Tải House Prices (python py30/download_data.py house). Dùng các cột số (điền thiếu bằng median), dự đoán log(SalePrice) bằng RandomForestRegressor(n_estimators=300, oob_score=True, random_state=0). So RMSE OOB (tính từ oob_prediction_) với RMSE 5-fold KFold(shuffle=True, random_state=0). Liệt kê 10 cột quan trọng nhất theo feature_importances_ và theo permutation_importance trên fold validation; ghi chỗ hai cách xếp hạng khác nhau.